# 4. Plots

RadDB draws four things. Each one **draws a single plot into a single Axes** and
returns the matplotlib artist, so you compose panels yourself by passing `ax=`.

| method | what it fixes | reads |
|---|---|---|
| `plot_ppi(sweep=...)` | one sweep, seen from above | horizontal gate faces |
| `plot_rhi(azimuth=...)` | one azimuth, all sweeps stacked | vertical gate faces |
| `plot_cappi(altitude=...)` | one altitude surface | vertical **and** horizontal faces |
| `plot_vcs(line=...)` | an arbitrary vertical slice | the cross-section geometry |

They all read the gate geometry from the LUT and join on `gate_id`, which means a
filtered, cropped or `sel`-ed RadDB **plots exactly the gates it holds** — nothing
is re-gridded onto a full azimuth x range mesh behind your back.

---

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import matplotlib.pyplot as plt
import raddb

# Keep the embedded figures small enough for GitHub to render this notebook.
#plt.rcParams["figure.dpi"] = 70

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — edit these three paths to point at your own data
# --------------------------------------------------------------------------
# ARCHIVE_DIR must be the same archive tutorial 1 wrote.  If it has not run,
# the cell below builds it.

MCH_DIR     = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/MCH_datatree").expanduser()
NEXRAD_DIR  = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

In [ ]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "L" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056).archive(datatree_dir=MCH_DIR)
else:
    print("archive already present:", ARCHIVE_DIR)

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
rdf = db.open(radars="L")
info = db.get_radar_info("L")
SITE = (info["longitude"], info["latitude"])
print(f"{len(rdf):,} gates | variables: {rdf.columns()}")

## 1. PPI

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.4))
rdf.plot_ppi(sweep=4, variable="DBZH", timestep="2024-06-08", ax=ax)
plt.show()

## 2. RHI — one azimuth, all sweeps

The RHI stacks every sweep along one azimuth, so you see the vertical structure
of the beam fan.

In [ ]:
fig, ax = plt.subplots()
rdf.plot_rhi(azimuth=90, variable="DBZH", timestep="2024-06-12", ax=ax)
plt.show()

## 3. CAPPI — a constant-altitude surface

A CAPPI takes the gates whose beam crosses a given altitude. Because the beam
climbs with range, that means different sweeps at different distances — which is
why a CAPPI usually covers more area than any single sweep.

In [ ]:
fig, ax = plt.subplots()
rdf.plot_cappi(altitude=2000, variable="DBZH", timestep="2024-06-12", ax=ax)
plt.tight_layout()
plt.show()

## 4. Vertical cross-section

A PPI has its sweep and an RHI its azimuth; a cross-section needs a **line**.
Either cut it first with `extract_cross_section` (tutorial 3) and then draw it:

In [ ]:
cs = rdf.extract_cross_section(p1=(SITE[0] - 0.6, SITE[1] - 0.35),
                               p2=(SITE[0] + 0.6, SITE[1] + 0.35),
                               crs=4326)

fig, ax = plt.subplots(figsize=(8, 4.5))
cs.plot_vcs(variable="DBZH", timestep="2024-06-12", ax=ax)
plt.show()

... or hand the line straight to `plot_vcs`, which cuts and draws in one step:

```python
rdf.plot_vcs(line=[(lon1, lat1), (lon2, lat2)], crs=4326)
rdf.plot_vcs(line="my_section.geojson")           # a LineString from a file
```

Passing a line to something that **already** carries a section is an error, and so
is drawing a section from data that has none — a cross-section has to be defined
exactly once.

## 5. Coordinates

`coords` controls the horizontal frame of the map-like plots:

| value | axes |
|---|---|
| `"xy"` *(default)* | metres east/north of the radar |
| `"lonlat"` | degrees |
| `"projected"` | the archive's own CRS (`x_2056` / `y_2056` here) |
| an EPSG int | that projection |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, coords in zip(axes, ["xy", "lonlat", "projected"]):
    rdf.plot_ppi(sweep=1, ax=ax, coords=coords)
    ax.set_title(f"coords={coords!r}")
plt.tight_layout()
plt.show()

`context=True` adds cartopy borders and coastlines. Projection and basemap are
independent — you choose the frame with `coords`, the background with `context`.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.4))
try:
    rdf.plot_ppi(sweep=1, ax=ax, coords="lonlat", context=True)
except Exception as exc:
    ax.set_title(f"cartopy unavailable: {type(exc).__name__}")
plt.show()

## 6. Any variable, any subset

`variable=` takes any column the data holds — including one you computed with
`add_feature` (tutorial 2).

In [ ]:
moments = [v for v in ("DBZH", "ZDR", "RHOHV", "PHIDP") if v in rdf.columns()]

fig, axes = plt.subplots(1, len(moments), figsize=(4.3 * len(moments), 4.0))
for ax, var in zip(axes, moments):
    rdf.plot_ppi(sweep=1, variable=var, ax=ax)
    ax.set_title(var)
plt.tight_layout()
plt.show()

In [ ]:
# Filter and crop first — the plot follows the data, gate for gate
sub = (rdf.filter({"var": "DBZH", "logic": ">", "threshold": 30})
          .crop_around_point(point=SITE, distance=50_000, crs=4326))

fig, ax = plt.subplots(figsize=(6.2, 5.4))
art = sub.plot_ppi(sweep=1, ax=ax)
ax.set_title(f"DBZH > 30 dBZ within 50 km — {len(art.get_paths()):,} gates drawn")
plt.show()

## 7. Composing a figure

Because each method fills one Axes, a multi-panel figure is ordinary matplotlib.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 9))
rdf.plot_ppi(sweep=1, ax=axes[0][0])
rdf.plot_rhi(azimuth=90, ax=axes[0][1])
rdf.plot_cappi(altitude=3000, ax=axes[1][0])
cs.plot_vcs(ax=axes[1][1])
fig.suptitle("radar L — PPI, RHI, CAPPI, cross-section", fontsize=15)
plt.tight_layout()
plt.show()

## 8. Plotting without an archive

The same functions accept a raw **DataTree**, computing the geometry from its own
coordinates. Handy for a quick look at a volume you have not archived yet.

The exception is `plot_vcs`: the cross-section path is `gate_id`-keyed, so it
needs an archive.

In [ ]:
dt = raddb.open_any_datatree(sorted(MCH_DIR.glob("L_*.zarr"))[0])

fig, ax = plt.subplots(figsize=(6.2, 5.4))
raddb.plot_ppi(dt, sweep=1, variable="DBZH", ax=ax)
ax.set_title("straight from a DataTree — no archive")
plt.show()

## 9. Saving

Pass `save="path.png"`, or use matplotlib directly. For vector output on a big
sweep, `rasterized=True` keeps the polygons as pixels inside the PDF and the file
small.

In [ ]:
out = ARCHIVE_DIR / "ppi_example.png"
fig, ax = plt.subplots(figsize=(6.2, 5.4))
rdf.plot_ppi(sweep=1, ax=ax, rasterized=True)
fig.savefig(out, dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved:", out, f"({out.stat().st_size / 1e3:.0f} kB)")

---
## Recap

```python
rdf.plot_ppi(sweep=1, variable="DBZH")
rdf.plot_rhi(azimuth=90)
rdf.plot_cappi(altitude=3000)
rdf.plot_vcs(line=[(lon1, lat1), (lon2, lat2)], crs=4326)
```

Shared arguments: `variable`, `radar`, `timestep`, `start_time` / `end_time`,
`coords`, `context`, `ax`, `save`, `rasterized`, and anything else is forwarded to
matplotlib.

A few things worth remembering:

- One plot, one Axes — **you** compose the figure with `ax=`.
- Geometry always comes from the LUT, so what you filtered is what you see.
- Beam width is a property of the **archive**, fixed when the LUT was generated
  (`beamwidth_deg` in `info.yaml`), not a plot argument.
- For a huge sweep, `rasterized=True` before saving to PDF.

---
That is the whole workflow: **archive → open & filter → crop → plot.**